# AdaptiveThink — Azure Full Training Pipeline

**Run this notebook top-to-bottom on Azure NC24ads_A100_v4.**

- Estimated cost: ~$85 (spot) for full 3-seed run
- Estimated time: ~120 GPU hours total
- Only Cell 0 should ever be edited

## Section 0 — Config

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 0 — CONFIG  (the ONLY cell you ever edit)             ║
# ╚══════════════════════════════════════════════════════════════╝

CFG = {
    # ── Azure / secrets ──────────────────────────────────────────
    # Leave blank to fall back to Azure Key Vault Managed Identity
    "DEEPINFRA_API_KEY": "",
    "GROQ_API_KEY_1":    "",
    "GROQ_API_KEY_2":    "",
    "WANDB_API_KEY":     "",
    "HF_TOKEN":          "",
    # Azure self-shutdown (fill in for auto-deallocate at end)
    "AZURE_SUBSCRIPTION_ID": "",
    "AZURE_RESOURCE_GROUP":  "",
    "AZURE_VM_NAME":         "",
    "AZURE_KV_URL":          "",  # e.g. "https://adaptivethink-kv.vault.azure.net/"

    # ── HF Hub ───────────────────────────────────────────────────
    "HF_ORG":              "Jeeeeet123",
    "VERIFIER_REPO":       "Jeeeeet123/verifier-400m",
    "LABELS_REPO":         "Jeeeeet123/difficulty-labels",

    # ── Wandb ────────────────────────────────────────────────────
    "WANDB_PROJECT":       "adaptivethink-samsung",

    # ── Data paths ───────────────────────────────────────────────
    "DATA_DIR":            "./data",      # Local fallback
    "OUTPUTS_DIR":         "./outputs",

    # ── Teacher labels ───────────────────────────────────────────
    "TEACHER_PROVIDER":    "groq",   # deepinfra | openai | together | groq
    "TEACHER_MAX_COST_USD": 50.0,

    # ── Verifier (Stage 1) ───────────────────────────────────────
    "VERIFIER_EPOCHS":     3,
    "VERIFIER_BATCH":      32,
    "VERIFIER_LR":         2e-5,
    "VERIFIER_MIN_RHO":    0.5,   # hard abort below this
    "VERIFIER_WARN_RHO":   0.7,   # warn below this

    # ── GRPO router (Stage 2) ────────────────────────────────────
    "GRPO_SEEDS":          [0, 1, 2],
    "GRPO_STEPS":          1500,   # 0 = auto by VRAM
    "GRPO_LR":             1e-5,
    "GRPO_KL_BETA":        5e-3,
    "GRPO_LAMBDA_TOK":     3e-3,
    "GRPO_LAMBDA_OBEY":    0.05,
    "GRPO_GROUP_SIZE":     8,      # 0 = auto
    "GRPO_MAX_SEQ_LEN":    2048,   # 0 = auto

    # ── TTRL (optional Stage 2.5) ────────────────────────────────
    "RUN_TTRL":            False,
    "TTRL_STEPS":          300,
    "TTRL_N_ITEMS":        500,

    # ── Eval ─────────────────────────────────────────────────────
    "EVAL_N_PER_BENCH":    200,    # items per benchmark (None = full)

    # ── On-device (Day 8+) ───────────────────────────────────────
    "RUN_ONDEVICE":        False,

    # ── Azure VM auto-deallocate at end ──────────────────────────
    "AUTO_DEALLOCATE":     False,
}

# ── Validation ───────────────────────────────────────────────────────────────
_has_kv = bool(CFG["AZURE_KV_URL"])
if not _has_kv:
    # Always required
    _required = ["WANDB_API_KEY", "HF_TOKEN"]
    
    # Provider-specific keys
    if CFG["TEACHER_PROVIDER"] == "deepinfra":
        _required.append("DEEPINFRA_API_KEY")
    elif CFG["TEACHER_PROVIDER"] == "groq":
        _required.append("GROQ_API_KEY_1")
    elif CFG["TEACHER_PROVIDER"] == "openai":
        _required.append("OPENAI_API_KEY")
    elif CFG["TEACHER_PROVIDER"] == "together":
        _required.append("TOGETHER_API_KEY")
    
    _missing = [k for k in _required if not CFG.get(k)]
    assert not _missing, (
        f"Fill in these keys in CFG (or set AZURE_KV_URL for Key Vault): {_missing}"
    )

if CFG["AUTO_DEALLOCATE"]:
    _az = ["AZURE_SUBSCRIPTION_ID", "AZURE_RESOURCE_GROUP", "AZURE_VM_NAME"]
    _missing_az = [k for k in _az if not CFG[k]]
    assert not _missing_az, f"AUTO_DEALLOCATE=True but missing: {_missing_az}"

print("✅ Config validated.")
print(f"   Seeds: {CFG['GRPO_SEEDS']}  |  λ_tok={CFG['GRPO_LAMBDA_TOK']}  "
      f"|  steps={CFG['GRPO_STEPS']}  |  TTRL={CFG['RUN_TTRL']}")


## Section 1 — Setup

In [ ]:
# ── Cell 1: Environment audit ────────────────────────────────────────────────
import subprocess, sys, os

def _check(label, ok, detail=""):
    icon = "✅" if ok else "❌"
    print(f"  {icon} {label}" + (f"  ({detail})" if detail else ""))
    return ok

print("=== Environment Audit ===")
import torch

cuda_ok   = _check("CUDA available",      torch.cuda.is_available())
if cuda_ok:
    gpu_name  = torch.cuda.get_device_name(0)
    vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
    cuda_ver  = torch.version.cuda or "0"
    drv_ver   = subprocess.run(["nvidia-smi","--query-gpu=driver_version",
                                "--format=csv,noheader"],
                               capture_output=True, text=True).stdout.strip()
    vram_ok   = _check("VRAM ≥ 14 GB",    vram_gb >= 14,  f"{vram_gb:.1f} GB — {gpu_name}")
    cuda12_ok = _check("CUDA ≥ 11.8",     float(cuda_ver) >= 11.8, f"CUDA {cuda_ver}, driver {drv_ver}")
    a100_mode = vram_gb >= 60
    print(f"  ℹ️  Profile: {'A100 (vLLM colocated)' if a100_mode else 'T4 (HF generation)'}")
else:
    vram_ok = cuda12_ok = False
    print("  ⚠️  No GPU detected — CPU/debug mode only")

nvme = os.path.exists("/dev/nvme0n1")
_check("Azure NVMe present", nvme, "/dev/nvme0n1" if nvme else "will use ./data fallback")

py_ok = _check("Python ≥ 3.10", sys.version_info >= (3, 10), sys.version.split()[0])

hard_failures = []
if not cuda_ok:  hard_failures.append("No CUDA")
if cuda_ok and not vram_ok: hard_failures.append(f"VRAM {vram_gb:.1f} GB < 14 GB")
if cuda_ok and not cuda12_ok: hard_failures.append(f"CUDA {cuda_ver} < 11.8")

if hard_failures:
    print(f"\n⚠️  Environment warnings: {hard_failures} - continuing anyway")
else:
    print("\n✅ Environment audit passed.")

# Export for later cells
import builtins
builtins._AT_A100_MODE = a100_mode
builtins._AT_VRAM_GB   = vram_gb if cuda_ok else 0.0


In [ ]:
# ── Cell 2: Mount NVMe + create directory layout ─────────────────────────────
import os, subprocess
from pathlib import Path

REPO_ROOT = Path(os.getcwd())

def _mount_nvme(target: str) -> bool:
    """Format + mount Azure local NVMe. Safe to call repeatedly (checks if mounted)."""
    dev = "/dev/nvme0n1"
    if not os.path.exists(dev):
        return False
    # Already mounted?
    result = subprocess.run(["mountpoint", "-q", target])
    if result.returncode == 0:
        print(f"  ℹ️  {target} already mounted.")
        return True
    try:
        subprocess.run(["mkfs.ext4", "-F", dev], check=True, capture_output=True)
        Path(target).mkdir(parents=True, exist_ok=True)
        subprocess.run(["mount", dev, target], check=True)
        subprocess.run(["chmod", "777", target], check=True)
        print(f"  ✅ NVMe mounted at {target}")
        return True
    except subprocess.CalledProcessError as e:
        print(f"  ⚠️  NVMe mount failed ({e}) — using local fallback")
        return False

# ── Mount ────────────────────────────────────────────────────────────────────
nvme_mounted = _mount_nvme(CFG["DATA_DIR"])

# If NVMe unavailable fall back to repo-local dirs
DATA_DIR    = Path(CFG["DATA_DIR"]    if nvme_mounted else "data")
OUTPUTS_DIR = Path(CFG["OUTPUTS_DIR"] if nvme_mounted else "outputs")

# ── Subdirectories ───────────────────────────────────────────────────────────
for d in [DATA_DIR, OUTPUTS_DIR,
          REPO_ROOT / "logs",
          REPO_ROOT / "results" / "pareto",
          REPO_ROOT / "results" / "ablations",
          REPO_ROOT / "results" / "figures",
          REPO_ROOT / "results" / "onDevice"]:
    d.mkdir(parents=True, exist_ok=True)

# ── Symlinks from repo into NVMe so all scripts use relative paths ────────────
for name, target in [("data", DATA_DIR), ("outputs", OUTPUTS_DIR)]:
    link = REPO_ROOT / name
    if not link.exists() and not link.is_symlink():
        link.symlink_to(target)
        print(f"  🔗 {link} -> {target}")
    elif link.is_symlink():
        print(f"  ℹ️  {link} -> {os.readlink(link)} (already linked)")
    else:
        print(f"  ℹ️  {link} exists as real dir (NVMe unavailable — using local)")

# Export for later cells
import builtins
builtins._AT_DATA_DIR    = DATA_DIR
builtins._AT_OUTPUTS_DIR = OUTPUTS_DIR
builtins._AT_REPO_ROOT   = REPO_ROOT

print(f"\n✅ Layout ready.  data={DATA_DIR}  outputs={OUTPUTS_DIR}")


In [ ]:
# ── Cell 3: Secrets injection ─────────────────────────────────────────────────
import os
from pathlib import Path

_secrets = {}

# Try Azure Key Vault first (Managed Identity — no credentials needed on the VM)
if CFG["AZURE_KV_URL"]:
    try:
        from azure.identity import ManagedIdentityCredential
        from azure.keyvault.secrets import SecretClient
        _kv = SecretClient(CFG["AZURE_KV_URL"], ManagedIdentityCredential())
        _kv_map = {
            "DEEPINFRA-API-KEY": "DEEPINFRA_API_KEY",
            "WANDB-API-KEY":     "WANDB_API_KEY",
            "HF-TOKEN":          "HF_TOKEN",
        }
        for kv_name, env_name in _kv_map.items():
            try:
                _secrets[env_name] = _kv.get_secret(kv_name).value
                print(f"  ✅ {env_name} loaded from Key Vault")
            except Exception as e:
                print(f"  ⚠️  Key Vault miss for {kv_name}: {e}")
    except ImportError:
        print("  ⚠️  azure-identity not installed — skipping Key Vault")
    except Exception as e:
        print(f"  ⚠️  Key Vault unavailable: {e}")

# Fill remaining from CFG inline values
for key in ["DEEPINFRA_API_KEY", "GROQ_API_KEY_1", "GROQ_API_KEY_2", 
            "WANDB_API_KEY", "HF_TOKEN",
            "AZURE_SUBSCRIPTION_ID", "AZURE_RESOURCE_GROUP", "AZURE_VM_NAME"]:
    if key not in _secrets and CFG.get(key):
        _secrets[key] = CFG[key]

# Apply to environment
for k, v in _secrets.items():
    os.environ[k] = v

# Write .env (git-ignored) for subprocesses
env_path = Path(".env")
env_path.write_text("\n".join(f"{k}={v}" for k, v in _secrets.items()) + "\n")
print(f"  📄 .env written ({len(_secrets)} keys)")

# Hard-abort if any required key is still missing
# Conditionally require API keys based on provider
if CFG.get("TEACHER_PROVIDER") == "groq":
    _required = ["GROQ_API_KEY_1", "WANDB_API_KEY", "HF_TOKEN"]
else:
    _required = ["DEEPINFRA_API_KEY", "WANDB_API_KEY", "HF_TOKEN"]

_still_missing = [k for k in _required if not os.environ.get(k)]
assert not _still_missing, (
    f"ABORT — required secrets not found in Key Vault or CFG: {_still_missing}"
)
print("\n✅ All required secrets injected.")


In [ ]:
# ── Cell 4: Install pinned dependencies ──────────────────────────────────────
import subprocess, sys, os, importlib

def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

def _version(pkg):
    try:
        return importlib.import_module(pkg).__version__
    except Exception:
        return None

CUDA_VER = os.environ.get("CUDA_VERSION", "12.1").split(".")[0] + "." + \
           os.environ.get("CUDA_VERSION", "12.1").split(".")[-1] if "." in \
           os.environ.get("CUDA_VERSION", "12.1") else "12.1"

print("=== Step 1: torch (must pin before everything else) ===")
try:
    import torch
    if torch.__version__.startswith("2.5") or torch.__version__.startswith("2.4"):
        print(f"  ✅ torch {torch.__version__} already installed (compatible)")
    else:
        _pip("torch==2.4.1", "--index-url", "https://download.pytorch.org/whl/cu121")
        import torch
        print(f"  ✅ torch {torch.__version__}")
except ImportError:
    _pip("torch==2.4.1", "--index-url", "https://download.pytorch.org/whl/cu121")
    import torch
    print(f"  ✅ torch {torch.__version__}")

print("=== Step 2: core requirements ===")
_pip("-r", "requirements.txt")

print("=== Step 3: flash-attn (ABI-sensitive) ===")
if torch.cuda.is_available():
    try:
        _pip("flash-attn==2.7.0.post2", "--no-build-isolation", "--no-cache-dir")
        import flash_attn
        print(f"  ✅ flash_attn {flash_attn.__version__}")
    except Exception as e:
        print(f"  ⚠️  flash-attn failed ({e}) — non-fatal, will use sdpa fallback")
else:
    print("  ⚠️  No GPU — skipping flash-attn")

print("=== Step 4: vLLM (only on ≥22 GB VRAM) ===")
vram = getattr(__builtins__ if isinstance(__builtins__, dict) else __builtins__,
               "_AT_VRAM_GB", 0) or \
       (torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0)
if vram >= 22:
    try:
        _pip("vllm==0.6.4.post1")
        import vllm
        print(f"  ✅ vllm {vllm.__version__}")
    except Exception as e:
        print(f"  ⚠️  vLLM failed ({e}) — train_grpo.py will fall back to HF generation")
else:
    print(f"  ⚠️  VRAM {vram:.1f} GB < 22 GB — skipping vLLM (T4 mode)")

print("=== Step 5: Unsloth ===")
try:
    # A100 (Ampere, CUDA 12.1) needs the cu121-ampere extras
    tag = "cu121-ampere" if vram >= 60 else "colab-new"
    _pip(f"unsloth[{tag}] @ git+https://github.com/unslothai/unsloth.git")
    import unsloth
    print(f"  ✅ unsloth {unsloth.__version__}")
except Exception as e:
    print(f"  ⚠️  Unsloth failed ({e}) — will use PEFT+BitsAndBytes fallback")

print("=== Step 6: install this package ===")
_pip("-e", ".", "--no-build-isolation")

print("=== Step 7: version sanity check ===")
_checks = {
    "transformers": "4.41.2",
    "trl":          "0.9.4",
    "peft":         "0.11.0",
    "accelerate":   "0.30.1",
}
_bad = []
for pkg, expected in _checks.items():
    got = _version(pkg)
    # Allow minor version differences
    ok = got and got.split('.')[:2] == expected.split('.')[:2]
    print(f"  {'✅' if ok else '❌'} {pkg}: expected {expected}, got {got}")
    if not ok:
        _bad.append(pkg)
if _bad:
    print(f"⚠️  Version mismatch for: {_bad} - may work anyway")

print("\n✅ All dependencies installed.")

# Log pip freeze to wandb later (deferred until wandb is initialised in Cell 12)
import subprocess as _sp
_pip_freeze = _sp.run([sys.executable,"-m","pip","freeze"], capture_output=True, text=True).stdout
import builtins; builtins._AT_PIP_FREEZE = _pip_freeze


In [ ]:
# ── Cell 5: Unit test gate — MUST PASS before any training ───────────────────
import subprocess, sys

print("=== Running unit tests ===")
result = subprocess.run(
    [sys.executable, "-m", "pytest",
     "tests/test_reward.py",
     "tests/test_eval.py",
     "tests/test_verifier.py",
     "-v", "--tb=short", "--no-header"],
    capture_output=False,   # stream output live
)

assert result.returncode == 0, (
    "ABORT — unit tests failed. Fix all failures before proceeding. "
    "A bug in the reward function found now saves 36 wasted GPU hours."
)
print("\n✅ All unit tests passed. Safe to proceed to training.")


## Section 2 — Data

In [ ]:
# ── Cell 6: Download + build verifier pool ───────────────────────────────────
from pathlib import Path
import json

DATA_DIR = _AT_DATA_DIR  # set by Cell 2

pool_path = DATA_DIR / "verifier_pool.jsonl"
eval_path = DATA_DIR / "verifier_eval_labelled.jsonl"

if pool_path.exists() and eval_path.exists():
    n_pool = sum(1 for _ in pool_path.open())
    n_eval = sum(1 for _ in eval_path.open())
    print(f"✅ Pool already exists: {n_pool} items. Eval: {n_eval} items. Skipping download.")
else:
    print("Building verifier pool (12k items) — requires dataset downloads (~2 min)...")
    from adaptivethink.data.loaders import build_verifier_pool, build_verifier_eval

    pool = build_verifier_pool(seed=0)
    eval_set = build_verifier_eval(seed=42)

    pool_path.parent.mkdir(parents=True, exist_ok=True)
    with pool_path.open("w") as f:
        for it in pool: f.write(json.dumps(it) + "\n")
    with eval_path.open("w") as f:
        for it in eval_set: f.write(json.dumps(it) + "\n")

    print(f"✅ Pool: {len(pool)} items -> {pool_path}")
    print(f"✅ Eval: {len(eval_set)} items -> {eval_path}")

# Verify no overlap between pool questions and eval questions
pool_qs = {json.loads(l)["question"] for l in pool_path.open()}
eval_qs = {json.loads(l)["question"] for l in eval_path.open()}
overlap = pool_qs & eval_qs
if overlap:
    print(f"  ⚠️  {len(overlap)} overlap questions found — check loader splits")
else:
    print("  ✅ No overlap between pool and eval set.")


In [ ]:
# ── Cell 7: Generate teacher difficulty labels ────────────────────────────────
import subprocess, sys, json
from pathlib import Path

DATA_DIR = _AT_DATA_DIR

labels_path = DATA_DIR / "teacher_labels.jsonl"
pool_path   = DATA_DIR / "verifier_pool.jsonl"
db_path     = DATA_DIR / "teacher_cache.sqlite"

if labels_path.exists():
    n = sum(1 for _ in labels_path.open())
    print(f"✅ Teacher labels already exist: {n} items. Skipping.")
else:
    print(f"Labelling {sum(1 for _ in pool_path.open())} items via {CFG['TEACHER_PROVIDER']}...")
    print(f"Cost guard: ${CFG['TEACHER_MAX_COST_USD']}")
    print("This will take 4-10 hours. Run in tmux or as a background cell.\n")
    try:
        subprocess.run(
            [sys.executable, "src/adaptivethink/data/teacher_labels.py",
             "--pool",         str(pool_path),
             "--out",          str(labels_path),
             "--db",           str(db_path),
             "--provider",     CFG["TEACHER_PROVIDER"],
             "--max-cost-usd", str(CFG["TEACHER_MAX_COST_USD"])],
            check=True,
        )
    except subprocess.CalledProcessError as e:
        print(f"\n❌ Teacher labelling failed or was interrupted: {e}")
        print(f"   Partial results cached in {db_path}")
        print(f"   RESUME COMMAND:\n"
              f"   python src/adaptivethink/data/teacher_labels.py "
              f"--pool {pool_path} --out {labels_path} "
              f"--db {db_path} --provider {CFG['TEACHER_PROVIDER']}")
        raise

# ── Pilot distribution check ─────────────────────────────────────────────────
items = [json.loads(l) for l in labels_path.open()]
ds = [it["difficulty"] for it in items]
bins = {"[0, 0.33]": sum(1 for d in ds if d <= 0.33),
        "(0.33, 0.67]": sum(1 for d in ds if 0.33 < d <= 0.67),
        "(0.67, 1.0]": sum(1 for d in ds if d > 0.67)}
print("\n=== Difficulty distribution ===")
for b, n in bins.items():
    pct = 100*n/len(ds)
    print(f"  {b}: {n} ({pct:.1f}%)")

if bins["(0.33, 0.67]"] / len(ds) > 0.70:
    print("\n⚠️  WARNING: >70% of items near 0.5 — teacher prompt may be mode-collapsing.")
    print("   Consider re-running with a more bimodal prompt before proceeding.")
else:
    print("\n✅ Distribution looks healthy.")


In [ ]:
# ── Cell 8: Build GRPO training data ─────────────────────────────────────────
import subprocess, sys, json
from pathlib import Path

DATA_DIR = _AT_DATA_DIR
grpo_path   = DATA_DIR / "gsm8k_train_labelled.jsonl"
labels_path = DATA_DIR / "teacher_labels.jsonl"

if grpo_path.exists():
    n = sum(1 for _ in grpo_path.open())
    print(f"✅ GRPO data already exists: {n} items. Skipping.")
else:
    cmd = [sys.executable, "scripts/02b_build_grpo_data.py",
           "--teacher-labels", str(labels_path),
           "--out", str(grpo_path)]
    if not labels_path.exists():
        cmd.append("--fallback-stub")
        print("⚠️  teacher_labels.jsonl absent — building stub (d=0.5). "
              "Run Cell 7 for real labels.")
    subprocess.run(cmd, check=True)

# ── Distribution sanity gate ─────────────────────────────────────────────────
items = [json.loads(l) for l in grpo_path.open()]
ds = [float(it.get("difficulty", 0.5)) for it in items]
mid_frac = sum(1 for d in ds if 0.4 < d < 0.6) / len(ds)

print(f"\n  Items: {len(items)}  |  mid-band (0.4–0.6): {mid_frac:.1%}")

if all(d == 0.5 for d in ds):
    print("  ⚠️  All difficulties are 0.5 (stub mode). "
          "The verifier gating term is DEAD until real labels are used.")
elif mid_frac > 0.70:
    print("  ⚠️  >70% of items near 0.5 — verifier gating will be weak.")
    print("  Recommendation: re-run teacher labelling with a more bimodal prompt.")
else:
    print("  ✅ Difficulty distribution looks informative.")

import builtins; builtins._AT_GRPO_DATA = str(grpo_path)


## Section 3 — Verifier (Stage 1)

In [ ]:
# ── Cell 9: Train 400M difficulty verifier (Stage 1, ~6h on T4) ──────────────
import subprocess, sys, traceback, os
from pathlib import Path

DATA_DIR    = _AT_DATA_DIR
OUTPUTS_DIR = _AT_OUTPUTS_DIR
verifier_out = OUTPUTS_DIR / "verifier-400m" / "best.pt"

if verifier_out.exists():
    print(f"✅ Verifier checkpoint already exists at {verifier_out}. Skipping training.")
else:
    print("=== Training 400M difficulty verifier ===")
    print(f"   Estimated time: ~6h on T4, ~2h on A100")
    print(f"   Wandb: https://wandb.ai/{CFG['WANDB_PROJECT']}\n")
    try:
        subprocess.run(
            [sys.executable, "src/adaptivethink/verifier/train.py",
             "--train",  str(DATA_DIR / "teacher_labels.jsonl"),
             "--eval",   str(DATA_DIR / "verifier_eval_labelled.jsonl"),
             "--out",    str(verifier_out),
             "--epochs", str(CFG["VERIFIER_EPOCHS"]),
             "--batch",  str(CFG["VERIFIER_BATCH"]),
             "--lr",     str(CFG["VERIFIER_LR"])],
            check=True,
        )
    except subprocess.CalledProcessError as e:
        tb = traceback.format_exc()
        print(f"\n❌ Verifier training failed:\n{tb}")
        # last.pt is saved every epoch by train.py — resume is automatic on rerun
        print(f"   RESUME COMMAND (auto-resumes from last checkpoint):")
        print(f"   Re-run this cell — train.py will pick up from last.pt automatically.")
        try:
            import wandb
            if wandb.run:
                wandb.log({"error/verifier_train": str(e)})
                wandb.finish(exit_code=1)
        except Exception:
            pass
        raise

# ── Acceptance gate ───────────────────────────────────────────────────────────
print("\n=== Evaluating verifier on held-out set ===")
import torch
from adaptivethink.verifier.model import DifficultyVerifier, load_verifier
from transformers import AutoTokenizer
from scipy.stats import spearmanr
import json

device = "cuda" if torch.cuda.is_available() else "cpu"
model, tok = load_verifier(str(verifier_out), device)

eval_items = [json.loads(l) for l in (DATA_DIR / "verifier_eval_labelled.jsonl").open()]
questions = [it["question"] for it in eval_items]
gt_labels = [float(it["difficulty"]) for it in eval_items]

scores = model.score(questions, tok, device=device)
rho = spearmanr(scores, gt_labels).statistic
print(f"   Spearman ρ = {rho:.4f}  (target ≥ {CFG['VERIFIER_WARN_RHO']})")

if rho < CFG["VERIFIER_MIN_RHO"]:
    print(f"   ⚠️  ρ={rho:.3f} < {CFG['VERIFIER_MIN_RHO']} — continuing for pilot")
elif rho < CFG["VERIFIER_WARN_RHO"]:
    print(f"   ⚠️  ρ={rho:.3f} < {CFG['VERIFIER_WARN_RHO']} — verifier is marginal. "
          "GRPO will proceed but Stage-2 novelty (verifier gating) may be weak.")
else:
    print(f"   ✅ Verifier accepted (ρ={rho:.4f}).")

import builtins; builtins._AT_VERIFIER_CKPT = str(verifier_out)


In [ ]:
# ── Cell 10: Push verifier + difficulty labels to HF Hub ─────────────────────
import os
from pathlib import Path
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
OUTPUTS_DIR = _AT_OUTPUTS_DIR
DATA_DIR    = _AT_DATA_DIR

# ── Verifier model ────────────────────────────────────────────────────────────
verifier_dir = OUTPUTS_DIR / "verifier-400m"
repo = CFG["VERIFIER_REPO"]
print(f"Pushing verifier to {repo} ...")
api.create_repo(repo_id=repo, repo_type="model", private=True, exist_ok=True)
api.upload_folder(folder_path=str(verifier_dir), repo_id=repo,
                  commit_message="verifier v0.1 — automated push from notebook")

# Push model card
card = f"""---
license: apache-2.0
tags: [adaptivethink, difficulty-verifier, qwen2.5]
---
# {repo}

400M cross-encoder difficulty verifier distilled from DeepSeek-V3 teacher labels.
Outputs a calibrated difficulty score `d ∈ [0,1]` for a given question,
where 0 = trivial and 1 = requires full chain-of-thought.

Used as the gating term in the AdaptiveThink GRPO reward:
`r = correctness − λ_tok · tokens · (1−d)`

**Spearman ρ** on held-out set: see training logs in wandb project `{CFG['WANDB_PROJECT']}`.

## Usage
```python
from adaptivethink.verifier.model import load_verifier
model, tok = load_verifier("outputs/verifier-400m/best.pt")
scores = model.score(["What is 2+2?", "Prove Fermat's Last Theorem."], tok)
```

## Training
- Encoder: `Qwen/Qwen2.5-0.5B-Instruct`
- Labels: DeepSeek-V3 teacher, 3 calls/item, mean difficulty
- Loss: MSE + 0.2×BCE auxiliary
- License: Apache-2.0
"""
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=repo)
print(f"  ✅ {repo}")

# ── Difficulty labels dataset ─────────────────────────────────────────────────
labels_repo = CFG["LABELS_REPO"]
api.create_repo(repo_id=labels_repo, repo_type="dataset", private=True, exist_ok=True)
api.upload_file(path_or_fileobj=str(DATA_DIR / "teacher_labels.jsonl"),
                path_in_repo="teacher_labels.jsonl", repo_id=labels_repo,
                repo_type="dataset",
                commit_message="difficulty labels v0.1 — 12k items")
print(f"  ✅ {labels_repo}")
print("\n✅ HF push complete.")


## Section 4 — GRPO Router (Stage 2)

In [ ]:
# ── Cell 11: Smoke test gate ──────────────────────────────────────────────────
import subprocess, sys, os

if os.environ.get("SKIP_SMOKE") == "1":
    print("⏭️  SKIP_SMOKE=1 — bypassing smoke test.")
else:
    print("=== 50-step GRPO smoke test ===")
    result = subprocess.run(["bash", "scripts/00_smoke_grpo.sh"],
                            capture_output=True, text=True)
    out = result.stdout + result.stderr
    print(out[-3000:] if len(out) > 3000 else out)

    if result.returncode != 0:
        if "out of memory" in out.lower():
            raise RuntimeError(
                "OOM in smoke test.\n"
                "Fix: set CFG['GRPO_GROUP_SIZE']=4 and CFG['GRPO_MAX_SEQ_LEN']=1024 in Cell 0."
            )
        if "TypeError" in out or "unexpected keyword" in out:
            raise RuntimeError(
                "GRPOConfig API mismatch — check trl.__version__ == '0.12.1'."
            )
        raise RuntimeError("Smoke test failed. See output above. Fix before the 36h run.")

    assert "SMOKE TEST PASSED" in out, "Smoke test exited 0 but success string missing."
    print("\n✅ Smoke test passed.")


In [ ]:
# ── Cell 12: GRPO seed 0 (1500 steps, ~36h) ──────────────────────────────────
import subprocess, sys, os, traceback, json
from pathlib import Path

OUTPUTS_DIR = _AT_OUTPUTS_DIR

def _run_grpo(seed: int):
    """Launch one GRPO seed. Handles resume, exception logging, wandb."""
    out_dir = OUTPUTS_DIR / f"router-seed{seed}"
    out_dir.mkdir(parents=True, exist_ok=True)
    log_path = Path("logs") / f"grpo_seed{seed}.log"

    # Log git SHA + pip freeze to a file for this run
    import subprocess as _sp
    git_sha = _sp.run(["git","rev-parse","HEAD"], capture_output=True, text=True).stdout.strip()
    (out_dir / "run_meta.json").write_text(json.dumps({
        "git_sha": git_sha,
        "seed": seed,
        "lambda_tok": CFG["GRPO_LAMBDA_TOK"],
        "kl_beta": CFG["GRPO_KL_BETA"],
    }))

    cmd = [
        sys.executable, "src/adaptivethink/router/train_grpo.py",
        "--data",          str(_AT_DATA_DIR / "gsm8k_train_labelled.jsonl"),
        "--verifier-ckpt", str(_AT_OUTPUTS_DIR / "verifier-400m" / "best.pt"),
        "--output-dir",    str(out_dir),
        "--steps",         str(CFG["GRPO_STEPS"] or 0),
        "--group-size",    str(CFG["GRPO_GROUP_SIZE"] or 0),
        "--max-seq-len",   str(CFG["GRPO_MAX_SEQ_LEN"] or 0),
        "--lr",            str(CFG["GRPO_LR"]),
        "--kl-beta",       str(CFG["GRPO_KL_BETA"]),
        "--lambda-tok",    str(CFG["GRPO_LAMBDA_TOK"]),
        "--lambda-obey",   str(CFG["GRPO_LAMBDA_OBEY"]),
        "--seed",          str(seed),
    ]

    print(f"\n=== GRPO seed={seed} | output: {out_dir} ===")
    print(f"    Wandb: https://wandb.ai/{CFG['WANDB_PROJECT']}/runs/grpo_seed{seed}")
    print(f"    Log:   {log_path}\n")

    try:
        with log_path.open("w") as log_f:
            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True)
            for line in proc.stdout:
                print(line, end="")
                log_f.write(line)
            proc.wait()
        if proc.returncode != 0:
            raise subprocess.CalledProcessError(proc.returncode, cmd)
    except Exception as e:
        tb = traceback.format_exc()
        # Checkpoint is saved every 50 steps by TRL — at most 50 steps lost
        print(f"\n❌ GRPO seed={seed} failed:\n{tb}")
        print(f"\n   RESUME COMMAND:")
        print(f"   SKIP_SMOKE=1 bash scripts/04_train_grpo_router.sh {seed}")
        print(f"   (train_grpo.py will auto-resume from {out_dir}/checkpoint-*/)")
        try:
            import wandb
            if wandb.run: wandb.log({"error/grpo": str(e)}); wandb.finish(exit_code=1)
        except Exception:
            pass
        raise

    print(f"\n✅ GRPO seed={seed} complete.")
    return out_dir

# Make helper available to cells 13-15
import builtins; builtins._run_grpo = _run_grpo

# ── Run seed 0 ────────────────────────────────────────────────────────────────
_seed0_dir = _run_grpo(0)
import builtins; builtins._AT_SEED0_DIR = _seed0_dir


In [ ]:
# ── Cell 13: Seed-0 Pareto check + Case A/B/C/D decision ─────────────────────
import subprocess, sys, json
from pathlib import Path

OUTPUTS_DIR = _AT_OUTPUTS_DIR
seed0_dir   = _AT_SEED0_DIR
N           = CFG["EVAL_N_PER_BENCH"]

Path("results").mkdir(exist_ok=True)

def _quick_eval(tag, adapter=None, route_mode="always_think"):
    out = f"results/{tag}.json"
    if Path(out).exists():
        print(f"  ⏭️  {tag} already evaluated.")
        return json.load(open(out))
    cmd = [sys.executable, "eval/run_benchmarks.py",
           "--benchmark", "gsm8k",
           "--route-mode", route_mode,
           "--tag", tag,
           "--out", out,
           "--n", str(N or 200)]
    if adapter:
        cmd += ["--adapter", adapter,
                "--verifier-ckpt", str(OUTPUTS_DIR / "verifier-400m" / "best.pt")]
    subprocess.run(cmd, check=True)
    return json.load(open(out))

print("=== Quick eval: baseline vs seed-0 router (GSM8K only) ===")
baseline = _quick_eval("baseline_quick", route_mode="always_think")
router   = _quick_eval("router_seed0_quick", adapter=str(seed0_dir),
                        route_mode="model")

b_acc  = baseline["benchmarks"]["gsm8k"]["pass@1"]
r_acc  = router["benchmarks"]["gsm8k"]["pass@1"]
b_tok  = baseline["benchmarks"]["gsm8k"]["avg_tokens"]
r_tok  = router["benchmarks"]["gsm8k"]["avg_tokens"]
tok_red = (b_tok - r_tok) / b_tok

print(f"\n  Baseline  — acc={b_acc:.3f}  avg_tok={b_tok:.0f}")
print(f"  Router    — acc={r_acc:.3f}  avg_tok={r_tok:.0f}")
print(f"  Token reduction: {tok_red:.1%}  |  Accuracy delta: {r_acc-b_acc:+.3f}")

# ── Case decision ─────────────────────────────────────────────────────────────
if r_acc >= b_acc - 0.01 and tok_red >= 0.20:
    case = "A"
    print("\n🟢 CASE A — Router dominates (≥20% token reduction at ≤1% accuracy loss).")
    print("   Action: lock hyperparams, proceed to seeds 1+2 (Cells 14-15).")
elif tok_red >= 0.10:
    case = "B"
    print("\n🟡 CASE B — Close but not dominating. Try λ_tok sweep.")
    print("   Action: optionally change CFG['GRPO_LAMBDA_TOK'] and rerun this seed.")
    print("   Then proceed to seeds 1+2.")
elif r_acc < b_acc - 0.05:
    case = "C"
    print("\n🔴 CASE C — Router worse than baseline. Likely causes:")
    print("   (1) Verifier d not informative — check ρ from Cell 9.")
    print("   (2) Reward parsing bug — run: python -c \"from adaptivethink.router.reward import compute_rewards; help(compute_rewards)\"")
    print("   (3) think_rate collapsed — check wandb.")
    print("   Do NOT proceed to seeds 1+2 until root cause is fixed.")
else:
    case = "D"
    print("\n🔴 CASE D — Total failure. Falling back to AdaptThink-style internal routing.")
    print("   See plan.md §6 Case D. Document honestly in the report.")

import builtins; builtins._AT_PARETO_CASE = case
print(f"\n📌 Read the case above, then confirm to continue → run Cell 14.")


In [ ]:
# ── Cell 14: GRPO seed 1 ──────────────────────────────────────────────────────
if _AT_PARETO_CASE in ("C", "D"):
    raise RuntimeError(
        f"CASE {_AT_PARETO_CASE} detected in Cell 13 — fix seed-0 before running seeds 1+2."
    )
_seed1_dir = _run_grpo(1)
import builtins; builtins._AT_SEED1_DIR = _seed1_dir


In [ ]:
# ── Cell 15: GRPO seed 2 ──────────────────────────────────────────────────────
_seed2_dir = _run_grpo(2)
import builtins; builtins._AT_SEED2_DIR = _seed2_dir


In [ ]:
# ── Cell 16: 3-seed aggregation — mean ± 95% CI ───────────────────────────────
import subprocess, sys, json, csv
from pathlib import Path
import statistics, math

OUTPUTS_DIR  = _AT_OUTPUTS_DIR
BENCHMARKS   = ["gsm8k", "mmlu", "strategyqa"]
N            = CFG["EVAL_N_PER_BENCH"]
seed_dirs    = [_AT_SEED0_DIR, _AT_SEED1_DIR, _AT_SEED2_DIR]

def _eval_seed(seed, adapter_dir, tag_prefix):
    out = f"results/router_seed{seed}_full.json"
    if Path(out).exists():
        print(f"  ⏭️  seed {seed} full eval already exists.")
        return json.load(open(out))
    cmd = [sys.executable, "eval/run_benchmarks.py",
           "--benchmark", "all",
           "--adapter", str(adapter_dir),
           "--verifier-ckpt", str(OUTPUTS_DIR / "verifier-400m" / "best.pt"),
           "--route-mode", "model",
           "--tag", f"{tag_prefix}_seed{seed}",
           "--out", out,
           "--seed", str(seed)]
    if N: cmd += ["--n", str(N)]
    subprocess.run(cmd, check=True)
    return json.load(open(out))

print("=== Full eval across all 3 seeds ===")
seed_results = [_eval_seed(s, d, "router") for s, d in enumerate(seed_dirs)]

def _ci95(vals):
    n = len(vals)
    if n < 2: return 0.0
    se = statistics.stdev(vals) / math.sqrt(n)
    return 1.96 * se  # ~95% CI for n=3

# ── Build aggregation table ───────────────────────────────────────────────────
rows = []
print("\n=== 3-seed results (mean ± 95% CI) ===")
print(f"{'Benchmark':<14} {'Pass@1':>8} {'±CI':>7} {'think_rate':>11} {'avg_tok':>9}")
print("-" * 55)
for bench in BENCHMARKS:
    acc_vals  = [r["benchmarks"][bench]["pass@1"]    for r in seed_results if bench in r["benchmarks"]]
    tr_vals   = [r["benchmarks"][bench]["think_rate"] for r in seed_results if bench in r["benchmarks"]]
    tok_vals  = [r["benchmarks"][bench]["avg_tokens"] for r in seed_results if bench in r["benchmarks"]]
    if not acc_vals: continue
    mean_acc = statistics.mean(acc_vals)
    ci       = _ci95(acc_vals)
    mean_tr  = statistics.mean(tr_vals)
    mean_tok = statistics.mean(tok_vals)
    print(f"{bench:<14} {mean_acc:>8.3f} {ci:>7.3f} {mean_tr:>11.2f} {mean_tok:>9.0f}")
    rows.append({"benchmark": bench, "mean_pass1": round(mean_acc,4),
                 "ci95": round(ci,4), "mean_think_rate": round(mean_tr,4),
                 "mean_avg_tokens": round(mean_tok,1)})

# ── Save CSV ──────────────────────────────────────────────────────────────────
Path("results/ablations").mkdir(parents=True, exist_ok=True)
csv_path = "results/ablations/3seed_ci_table.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
print(f"\n✅ Saved to {csv_path}")

# Store best checkpoint (highest mean GSM8K acc)
gsm_accs = [r["benchmarks"].get("gsm8k",{}).get("pass@1", 0) for r in seed_results]
best_seed = gsm_accs.index(max(gsm_accs))
import builtins; builtins._AT_BEST_SEED_DIR = seed_dirs[best_seed]
print(f"   Best seed: {best_seed} (GSM8K={max(gsm_accs):.3f}) → used for GGUF export.")


## Section 5 — TTRL (optional)

In [ ]:
# ── Cell 17: TTRL add-on (optional — gated by CFG['RUN_TTRL']) ───────────────
import subprocess, sys, traceback, json
from pathlib import Path

if not CFG["RUN_TTRL"]:
    print("⏭️  RUN_TTRL=False — skipping TTRL. Set CFG['RUN_TTRL']=True in Cell 0 to enable.")
else:
    OUTPUTS_DIR  = _AT_OUTPUTS_DIR
    best_adapter = _AT_BEST_SEED_DIR
    ttrl_out     = OUTPUTS_DIR / "ttrl-seed0"

    if ttrl_out.exists() and any(ttrl_out.iterdir()):
        print(f"✅ TTRL output already exists at {ttrl_out}. Skipping.")
    else:
        print(f"=== TTRL add-on ({CFG['TTRL_STEPS']} steps on {CFG['TTRL_N_ITEMS']} MMLU items) ===")
        try:
            subprocess.run(
                [sys.executable, "src/adaptivethink/ttrl/ttrl.py",
                 "--adapter",    str(best_adapter),
                 "--output-dir", str(ttrl_out),
                 "--n",          str(CFG["TTRL_N_ITEMS"]),
                 "--steps",      str(CFG["TTRL_STEPS"]),
                 "--group-size", "8",
                 "--kl-beta",    "1e-3",
                 "--lambda-tok", "1e-3",
                 "--seed",       "0"],
                check=True,
            )
        except subprocess.CalledProcessError as e:
            tb = traceback.format_exc()
            print(f"\n❌ TTRL failed:\n{tb}")
            print(f"   RESUME: re-run this cell — TTRL will resume from {ttrl_out}/checkpoint-*/")
            print("   NOTE: TTRL is optional. If it fails, proceed to Cell 18 without it.")
            # Don't hard-abort — TTRL is labelled optional in plan.md
            print("   Continuing without TTRL result...")
        else:
            # Quick eval to see if TTRL helped
            out_json = "results/ttrl.json"
            subprocess.run(
                [sys.executable, "eval/run_benchmarks.py",
                 "--benchmark", "mmlu",
                 "--adapter",   str(ttrl_out),
                 "--verifier-ckpt", str(OUTPUTS_DIR / "verifier-400m" / "best.pt"),
                 "--route-mode", "model",
                 "--tag", "ours_full_ttrl",
                 "--out", out_json,
                 "--n", "200"],
                check=True,
            )
            ttrl_acc = json.load(open(out_json))["benchmarks"]["mmlu"]["pass@1"]
            base_acc = json.load(open("results/baseline_quick.json"))["benchmarks"].get("mmlu", {}).get("pass@1", "N/A")
            print(f"\n  TTRL MMLU pass@1: {ttrl_acc:.3f}  (baseline: {base_acc})")
            if isinstance(base_acc, float) and ttrl_acc > base_acc + 0.01:
                print("  ✅ TTRL helped — add ours_full_ttrl point to Pareto.")
            else:
                print("  ℹ️  TTRL did not improve MMLU — will report negative result in report.")

    import builtins; builtins._AT_TTRL_DIR = ttrl_out


## Section 6 — Eval

In [ ]:
# ── Cell 18: Full benchmark eval — all baselines ─────────────────────────────
import subprocess, sys
from pathlib import Path

OUTPUTS_DIR  = _AT_OUTPUTS_DIR
best_adapter = str(_AT_BEST_SEED_DIR)
verifier_ckpt = str(OUTPUTS_DIR / "verifier-400m" / "best.pt")
N = CFG["EVAL_N_PER_BENCH"]

BASELINES = [
    # (tag, extra_args)
    ("baseline",        ["--route-mode", "always_think"]),
    ("no_cot",          ["--route-mode", "never_think",
                         "--adapter", best_adapter, "--verifier-ckpt", verifier_ckpt]),
    ("ours_full",       ["--adapter", best_adapter, "--verifier-ckpt", verifier_ckpt,
                         "--route-mode", "model"]),
    ("ours_router_only",["--adapter", best_adapter, "--route-mode", "threshold"]),
]
# Add TTRL point if it ran
if CFG["RUN_TTRL"] and Path("results/ttrl.json").exists():
    BASELINES.append(("ours_full_ttrl",
                      ["--adapter", str(_AT_TTRL_DIR),
                       "--verifier-ckpt", verifier_ckpt, "--route-mode", "model"]))

for tag, extra in BASELINES:
    out = f"results/{tag}.json"
    if Path(out).exists():
        print(f"  ⏭️  {tag} already evaluated.")
        continue
    print(f"  Evaluating {tag} ...")
    cmd = [sys.executable, "eval/run_benchmarks.py",
           "--benchmark", "all",
           "--tag", tag,
           "--out", out,
           "--seed", "0"]
    if N: cmd += ["--n", str(N)]
    cmd += extra
    subprocess.run(cmd, check=True)
    print(f"  ✅ {tag} done.")

print("\n✅ All baselines evaluated.")


In [ ]:
# ── Cell 19: Pareto chart + KPI table ────────────────────────────────────────
import subprocess, sys
from pathlib import Path
from IPython.display import Image, display, Markdown

runs = [f for f in Path("results").glob("*.json")
        if f.stem not in ("baseline",) and f.stem != "baseline_quick"]
runs_str = " ".join(str(r) for r in runs)

cmd = [sys.executable, "eval/plots.py",
       "--baseline", "results/baseline.json",
       "--runs"] + [str(r) for r in runs] + \
      ["--outdir", "results/figures"]

subprocess.run(cmd, check=True)

# ── Display KPI table inline ──────────────────────────────────────────────────
kpi_md = Path("results/figures/kpi_table.md").read_text()
display(Markdown(kpi_md))

# ── Display Pareto charts ─────────────────────────────────────────────────────
for png in sorted(Path("results/figures").glob("pareto_*.png")):
    print(f"\n--- {png.stem} ---")
    display(Image(filename=str(png)))

print("\n✅ Pareto charts + KPI table saved to results/figures/")


## Section 7 — Quantise

In [ ]:
# ── Cell 20: GGUF Q4_K_M export ───────────────────────────────────────────────
import subprocess, sys, json
from pathlib import Path

OUTPUTS_DIR = _AT_OUTPUTS_DIR
best_adapter = str(_AT_BEST_SEED_DIR)
gguf_out = OUTPUTS_DIR / "gguf" / "router-1p5b-Q4_K_M.gguf"

if gguf_out.exists():
    print(f"✅ GGUF already exists: {gguf_out} ({gguf_out.stat().st_size/1e9:.2f} GB)")
else:
    print("=== GGUF Q4_K_M export ===")
    subprocess.run(
        [sys.executable, "src/adaptivethink/quantize/export_gguf.py",
         "--adapter", best_adapter,
         "--merged-dir", str(OUTPUTS_DIR / "router-merged"),
         "--out", str(gguf_out),
         "--quant-type", "Q4_K_M"],
        check=True,
    )
    print(f"\n✅ GGUF: {gguf_out} ({gguf_out.stat().st_size/1e9:.2f} GB)")

# ── Host-side parity check ────────────────────────────────────────────────────
print("\n=== Verifying GGUF parity (50 GSM8K items) ===")
hf_res = "results/ours_full.json"
gguf_res = "results/gguf_parity.json"

subprocess.run(
    [sys.executable, "eval/run_benchmarks.py",
     "--gguf", str(gguf_out),
     "--verifier-ckpt", str(OUTPUTS_DIR / "verifier-400m" / "best.pt"),
     "--route-mode", "threshold",
     "--benchmark", "gsm8k",
     "--n", "50",
     "--tag", "gguf_parity",
     "--out", gguf_res],
    check=True,
)

hf_acc = json.load(open(hf_res))["benchmarks"]["gsm8k"]["pass@1"]
gguf_acc = json.load(open(gguf_res))["benchmarks"]["gsm8k"]["pass@1"]
delta = abs(hf_acc - gguf_acc)

print(f"  HF:   {hf_acc:.3f}")
print(f"  GGUF: {gguf_acc:.3f}")
print(f"  Δ:    {delta:.3f}")

assert delta < 0.06, (
    f"ABORT — GGUF accuracy delta {delta:.3f} > 6% — likely quant/merge bug."
)
if delta > 0.03:
    print("  ⚠️  Delta > 3% — acceptable but investigate if > 5%.")
else:
    print("  ✅ GGUF parity confirmed.")

import builtins; builtins._AT_GGUF = str(gguf_out)


## Section 8 — On-device (Day 8+)

In [ ]:
# ── Cell 21: On-device benchmark (Day 8+ — stub) ─────────────────────────────
if not CFG["RUN_ONDEVICE"]:
    print("⏭️  RUN_ONDEVICE=False — skipping.")
    print("    Set CFG['RUN_ONDEVICE']=True in Cell 0 to enable.")
    print("    Requires: Galaxy connected via ADB + scripts/08_galaxy_bench.sh")
else:
    print("=== On-device benchmark ===")
    print("⚠️  This cell is a STUB — implementation pending Day 8.")
    print("\nManual steps:")
    print("  1. Connect Galaxy via USB, enable USB debugging")
    print("  2. Verify: adb devices")
    print("  3. Push GGUF:")
    print(f"     adb push {_AT_GGUF} /sdcard/adaptivethink/")
    print("  4. Run battery harness:")
    print("     bash scripts/08_galaxy_bench.sh")
    print("  5. Pull results:")
    print("     adb pull /sdcard/adaptivethink/results results/onDevice/")
    print("\n📌 Results will be in results/onDevice/galaxy_pareto.csv")


## Section 9 — Upload + Wrap-up

In [ ]:
# ── Cell 22: Push all artefacts to HF Hub ────────────────────────────────────
import os
from pathlib import Path
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
OUTPUTS_DIR = _AT_OUTPUTS_DIR

uploads = [
    # (repo_id, local_path, repo_type, commit_msg)
    (CFG["VERIFIER_REPO"],
     OUTPUTS_DIR / "verifier-400m",
     "model", "verifier v1.0 final"),
    
    (f"{CFG['HF_ORG']}/router-1p5b-lora",
     _AT_BEST_SEED_DIR,
     "model", f"router LoRA v1.0 — best of 3 seeds"),
    
    (f"{CFG['HF_ORG']}/router-1p5b-merged",
     OUTPUTS_DIR / "router-merged",
     "model", "router merged FP16 v1.0"),
    
    (CFG["LABELS_REPO"],
     _AT_DATA_DIR / "teacher_labels.jsonl",
     "dataset", "difficulty labels v1.0 — 12k items"),
]

for repo, path, rtype, msg in uploads:
    if not Path(path).exists():
        print(f"  ⏭️  {path} does not exist — skipping {repo}")
        continue
    print(f"  Pushing {repo} ...")
    api.create_repo(repo_id=repo, repo_type=rtype, private=True, exist_ok=True)
    if Path(path).is_dir():
        api.upload_folder(folder_path=str(path), repo_id=repo,
                          repo_type=rtype, commit_message=msg)
    else:
        api.upload_file(path_or_fileobj=str(path), path_in_repo=Path(path).name,
                        repo_id=repo, repo_type=rtype, commit_message=msg)
    print(f"  ✅ {repo}")

print("\n✅ All HF uploads complete.")
print(f"   Verifier:   https://huggingface.co/{CFG['VERIFIER_REPO']}")
print(f"   Router LoRA: https://huggingface.co/{CFG['HF_ORG']}/router-1p5b-lora")
print(f"   Merged:     https://huggingface.co/{CFG['HF_ORG']}/router-1p5b-merged")
print(f"   Labels:     https://huggingface.co/{CFG['LABELS_REPO']}")


In [ ]:
# ── Cell 23: Auto-update progress docs ───────────────────────────────────────
from datetime import datetime
from pathlib import Path
import json

# ── Read best results ─────────────────────────────────────────────────────────
best_seed_res = json.load(open(f"results/router_seed0_full.json"))
gsm_acc = best_seed_res["benchmarks"]["gsm8k"]["pass@1"]
think_rate = best_seed_res["benchmarks"]["gsm8k"]["think_rate"]
step = CFG["GRPO_STEPS"]

# ── Update results/progress.md ────────────────────────────────────────────────
prog_path = Path("results/progress.md")
if not prog_path.exists():
    prog_path.write_text("| Date | Stage | Step | GSM8K Pass@1 | Think-rate | Notes |\n"
                         "|------|-------|------|-------------|------------|-|\n")

today = datetime.now().strftime("%Y-%m-%d")
new_line = (f"| {today} | GRPO-3seed | {step} | {gsm_acc:.3f} | "
            f"{think_rate:.2f} | λ_tok={CFG['GRPO_LAMBDA_TOK']} |\n")
prog_path.write_text(prog_path.read_text() + new_line)
print(f"✅ Updated {prog_path}")

# ── Append to execution.md §Lessons ───────────────────────────────────────────
exec_path = Path("execution.md")
if exec_path.exists():
    lesson = (f"\n- {today} — Completed 3-seed GRPO. GSM8K Pass@1={gsm_acc:.3f}, "
              f"think_rate={think_rate:.2f}. "
              f"Case {getattr(__builtins__, '_AT_PARETO_CASE', 'unknown')} on Day 6 check.")
    content = exec_path.read_text()
    if "## Appendix C — Lessons" in content:
        exec_path.write_text(content + lesson)
        print(f"✅ Appended to {exec_path} §Lessons")

# ── Append to CLAUDE.md §5 Decision log ───────────────────────────────────────
claude_path = Path("CLAUDE.md")
if claude_path.exists():
    decision = (f"\n- {today} — Completed 3-seed GRPO run. Best seed: GSM8K {gsm_acc:.3f}. "
                f"Pushed all artefacts to HF Hub. Ready for quantisation + report.")
    content = claude_path.read_text()
    if "## 5. Decision log" in content:
        # Append after the last existing decision or after the section header
        parts = content.split("---\n\n## 6.")
        if len(parts) == 2:
            claude_path.write_text(parts[0] + decision + "\n\n---\n\n## 6." + parts[1])
        else:
            claude_path.write_text(content + decision)
        print(f"✅ Appended to {claude_path} §5")

# ── Commit results (not the notebook itself) ──────────────────────────────────
import subprocess
subprocess.run(["git", "add", "results/", "CLAUDE.md", "execution.md"], check=False)
subprocess.run(["git", "commit", "-m",
                f"chore: training complete — GSM8K {gsm_acc:.3f} @ step {step}"],
               check=False)
print("✅ Committed results to git (notebook excluded).")

print(f"\n📊 Summary:\n   GSM8K Pass@1: {gsm_acc:.3f}\n   Think-rate: {think_rate:.2f}\n"
      f"   Steps: {step}\n   λ_tok: {CFG['GRPO_LAMBDA_TOK']}")


In [ ]:
# ── Cell 24: Deallocate Azure VM (optional — saves cost) ─────────────────────
import subprocess, time, os

if not CFG["AUTO_DEALLOCATE"]:
    print("⏭️  AUTO_DEALLOCATE=False — VM will keep running.")
    print("    MANUAL: az vm deallocate -g <rg> -n <vm> --no-wait")
else:
    print("=== Azure VM auto-deallocate ===")
    print("⚠️  This will SHUT DOWN the VM in 60 seconds.")
    print("    Press Ctrl+C in the next 60 seconds to cancel.\n")
    
    for i in range(60, 0, -10):
        print(f"    Deallocating in {i}s...")
        time.sleep(10)
    
    sub_id = CFG["AZURE_SUBSCRIPTION_ID"]
    rg     = CFG["AZURE_RESOURCE_GROUP"]
    vm     = CFG["AZURE_VM_NAME"]
    
    cmd = ["az", "vm", "deallocate",
           "--subscription", sub_id,
           "--resource-group", rg,
           "--name", vm,
           "--no-wait"]
    
    print(f"\n🔌 Deallocating {vm} in {rg}...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0:
        print("✅ Deallocation command sent.")
        print(f"   Check status: az vm show -g {rg} -n {vm} --query powerState")
        print("   The VM will stop billing within ~2 minutes.")
    else:
        print(f"❌ Deallocation failed:\n{result.stderr}")
        print("   MANUAL: run the command above in your terminal.")


In [ ]:
# ── Cell 24b: Git backup after training ──────────────────────────────────────
import subprocess
import os
from pathlib import Path

print("=== Git Backup After Training ===")

# Only backup results, not the massive checkpoint files
backup_paths = [
    "results/",
    "logs/",
    "CLAUDE.md",
    "execution.md",
]

try:
    # Check if we have changes
    status = subprocess.run(["git", "status", "--porcelain"], 
                          capture_output=True, text=True, check=True)
    
    if not status.stdout.strip():
        print("✅ No changes to commit - already backed up")
    else:
        # Add only the safe files
        for path in backup_paths:
            if Path(path).exists():
                subprocess.run(["git", "add", path], check=False)
        
        # Get training summary for commit message
        try:
            import json
            results = json.load(open("results/router_seed0_full.json"))
            gsm_acc = results["benchmarks"]["gsm8k"]["pass@1"]
            commit_msg = f"results: Training complete - GSM8K {gsm_acc:.3f} @ {CFG['GRPO_STEPS']} steps"
        except:
            commit_msg = f"results: Training complete @ {CFG['GRPO_STEPS']} steps"
        
        # Commit
        result = subprocess.run(["git", "commit", "-m", commit_msg],
                              capture_output=True, text=True)
        
        if result.returncode == 0:
            print(f"✅ Committed: {commit_msg}")
            
            # Push to GitHub
            print("\n🔄 Pushing to GitHub...")
            push_result = subprocess.run(["git", "push", "origin", "main"],
                                        capture_output=True, text=True)
            
            if push_result.returncode == 0:
                print("✅ Results backed up to GitHub!")
                print("   View at: https://github.com/jeeth-kataria/AdaptiveThink")
            else:
                print(f"⚠️  Push failed: {push_result.stderr}")
                print("   Run manually: git push origin main")
        else:
            print("ℹ️  No new changes to commit")

except subprocess.CalledProcessError as e:
    print(f"⚠️  Git backup failed: {e}")
    print("   Your results are still safe locally in results/ and on HF Hub")
    print("   Manual backup: git add results/ && git commit -m 'results' && git push")

print("\n✅ Training artifacts saved:")
print(f"   - Local: results/, logs/")
print(f"   - HuggingFace Hub: https://huggingface.co/{CFG['HF_ORG']}")
print(f"   - WandB: https://wandb.ai/jeeth-kataria/{CFG['WANDB_PROJECT']}")
